In [1]:
import requests #to fetch data from API
import pandas as pd #for handling and anlysing data
import numpy as np #for numericsl operations
from sklearn.model_selection import train_test_split #to split data
from sklearn.preprocessing import LabelEncoder #to convert categorical data in to numerical values
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor #models for clasification AND REGRESSION
from sklearn.metrics import mean_squared_error #for finding accuracy
from datetime import datetime, timedelta #for working with date and time
import matplotlib.pyplot as plt #for visualization
import pytz



In [2]:
API_KEY = 'ab384ee816f8b1c604b436c9e0454990' #API key in weatherOpen API
BASE_URL = 'https://api.openweathermap.org/data/2.5/'

1.fetch current weather data


In [3]:
def get_current_weather(city):
  url = f"{BASE_URL}weather?q={city}&appid={API_KEY}&units=metric" #API req URL
  response = requests.get(url) #send the get req to the API
  data =  response.json()
  return {
    'city' : data['name'],
    'current_temp' : round(data['main']['temp']),
    'feels_like': round(data['main']['feels_like']),
    'temp_min' : round(data['main']['temp_min']),
    'temp_max' : round(data['main']['temp_max']),
    'humidity' : round(data['main']['humidity']),
    'description' : data['weather'][0]['description'],
    'country' : data['sys']['country'],
    'wind_gust_dir' : data['wind']['deg'],
    'pressure' : data['main']['pressure'],
    'wind_gust_speed' : data['wind']['speed'],
    

  }

2.Read Historical data


In [4]:
def read_historical_data(filename):
  df = pd.read_csv(filename) #load csv file
  df = df.dropna() #remove rows with missing values
  df = df.drop_duplicates()
  return df
  

3.Prepare data for training


In [5]:
def prepare_data(data):
  le  = LabelEncoder() #initialize label encoder
  data['WindGustDir'] = le.fit_transform(data['WindGustDir']) #fit and transform the data
  data['RainTomorrow'] = le.fit_transform(data['RainTomorrow']) #fit and transform the data

  X = data[['MinTemp','Maxtemp','WindGustDir', 'WindGustSpeed', 'Humidity','Pressure', 'Temp']] #feature variables
  y = data['RainTomorrow'] #target variable
  return X, y, le #returning feature variables, target variable and label encoder

4.Train Rain Prediciton Model


In [6]:
def train_rain_model(X,y):
  #split data
  X_train, X_test, y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42)
  model = RandomForestClassifier(n_estimators=100, random_state=42)
  model.fit(X_train, y_train)

  y_pred = model.predict(X_test) #to make predicitons on test set

  print("Mean Squared Error for Rain model ")
  print(mean_squared_error(y_test, y_pred))

  return model


5.Prepare regression data


In [7]:
def prepare_regression_data(data,feature):
  X,y = [], [] #initialize empty lists

  for i in range(len(data)-1):
    X.append(data[feature].iloc[i])
    y.append(data[feature].iloc[i+1])

  X = np.array(X).reshape(-1,1)
  y = np.array(y)
  return X,y

  

5.Train regression model


In [8]:
def train_regression_model(X,y):
  model = RandomForestRegressor(n_estimators=100, random_state=42)
  model.fit(X, y)
  return model

5.Predict Future

In [9]:
def predict_future(model, current_value):
  predictions = [current_value]

  for i in range(5):
    next_value = model.predict(np.array([[predictions[-1]]]))

    predictions.append(next_value[0])

    return predictions[1:]

6.Weather Analaysis


In [ ]:
def weather_view():
  city = input("Enter any city name: ")
  current_weather = get_current_weather(city)

  #load historical data
  historical_data = read_historical_data('weather.csv')

  #prepare and train rain prediciton model

  X,y,le = prepare_data(historical_data)

  rain_model = train_rain_model(X,y)


  #map wind direction to compass points
  wind_deg = current_weather['wind_gust_dir'] % 360
  compass_points = [
  ("N", 0, 11.25), ("NNE", 11.25, 33.75), ("NE", 33.75, 56.25), ("ENE", 56.25, 78.75), ("E", 78.75, 101.25), ("ESE", 101.25, 123.75), ("SE", 123.75, 146.25), ("SSE", 146.25, 168.75), ("S", 168.75, 191.25),
  ("SSW", 191.25, 213.75), ("SW", 213.75, 236.25), ("WSW", 236.25, 258.75), ("w", 258.75, 281.25), ("WNW", 281.25, 303.75), ("Ny", 303.75, 326.25),
  ("NNW", 326.25, 348.75)
  ]
  compass_direction = next(point for point, start, end in compass_points if start <= wind_deg < end)

  compass_direction_encoded = le.transform([compass_direction])[0] if compass_direction in le.classes_ else -1

    #prepare current data for prediction

  current_data = {
      'MinTemp' : current_weather['temp_min'],
      'Maxtemp' : current_weather['temp_max'],
      'WindGustDir' : compass_direction_encoded,
      'WindGustSpeed' : current_weather['wind_gust_speed'],
      'Humidity' : current_weather['humidity'],
      'Pressure' : current_weather['pressure'],
      'Temp' : current_weather['current_temp'],
      
    }
  current_df = pd.DataFrame([current_data])
  #rain prediction

  rain_prediction = rain_model.predict(current_df)[0]

  #prepare regression model for temperature and humidity
  X_temp,y_temp = prepare_regression_data(historical_data, 'Temp')

  X_hum, y_hum = prepare_regression_data(historical_data, 'Humidity')

  temp_model = train_regression_model(X_temp, y_temp)

  hum_model = train_regression_model(X_hum, y_hum)

  future_temp = predict_future(temp_model, current_weather['temp_min'])


  future_hum = predict_future(hum_model, current_weather['humidity'])

  #prepare time for future predicitons

  timezone = pytz.timezone('Asia/Chenna')
  now = datetime.now(timezone)
  next_hour = now + timedelta(hours=1)
  next_hour = next_hour.replace(minute=0, second=0, microsecond=0)

  future_times = [(next_hour + timedelta(hours=i)).strftime("%H:00") for i in range(5)]

  #display results
  print(f"city: {city}, {current_weather['country']}")
  print(f"Current Temperature: {current_weather['current_temp']}°C")
  print(f"Feels Like: {current_weather['feels_like']}")
  print(f"Minimum Temperature: {current_weather['temp_min']}°C")
  print(f"Maximum Temperature: {current_weather['temp_max']}°C")
  print(f"Humidity: {current_weather['humidity']}%")
  print(f"Rain Prediction: {'Yes' if rain_prediction else 'No'}")


  print("\nFuture Temperature Predictions:")

  for time, temp in zip(future_times, future_temp):
    print(f"{time}: {round(temp,1)}°C")

  print("\nFuture Humidity Predictions:")

  for time, humidity in zip(future_times, future_hum):
    print(f"{time}: {round(humidity,1)}%")




    







  
  

  
